<img src="../assets/stac2cube_logo.png" alt="stac2cube logo" width="260">

# Graphical User Interface Tools

Build a data cube from a STAC catalog, edit it, and refine it into an Analysis Ready Data (ARD) cube. All offered in interfaces, no coding.

**Author:** Baturalp Arisoy<br>
**Contact:** baturalp.arisoy@uni-wuerzburg.de - Call me Batu :)

- **1. Data Cube Builder** - Search a STAC catalog and build a cube for your area and dates
- **2. Data Cube Editor** - Open a cube: subset, clip, reproject, composite, update, mosaic and more
- **3. ARD Cube Tools** - s2cloudless cloud/shadow masking, co-registration, super-resolution
- **? Troubleshooting** - Potential errors and warnings you might encounter

Each interface is independent.

> **⚠️ Save your cubes as NetCDF or Zarr**
>
> The **Data Cube Editor** and the **ARD Cube Tools** can only open cubes stored as **NetCDF (`.nc`)** or **Zarr (`.zarr`)**. Geotiffs cannot be read back in.
>
> Write Geotiffs at the very end: **Data Cube Editor -> Export Options -> Geotiffs (Cloud Optimized)**.

<details>
<summary><b>How to STOP the processing?</b></summary>

Do any one of these:

- delete the interface output cell
- interrupt the kernel
- restart the kernel

</details>

# 1. Data Cube Builder

Query a STAC catalog and build the cube.

<details>
<summary><b>What is in this interface?</b></summary>

- **Basic / Advanced Parameters** - area, dates, mission, bands, cloud masking, and the rest of the build options
- **Data Source** - which STAC catalog to query (some sources need credentials)
- **Availability / Area Coverage results** - check what the catalog holds for your area before building
- **Result** - the cube that was built, with its dates
- **Temporal Composites** - reduce the time axis (mean, median, min, max, std)
- **Visualization** - interactive viewer and GIF animation
- **Export Options** - NetCDF, Zarr or Cloud Optimized Geotiffs

</details>

<details>
<summary><b>ℹ️ Interactive View here is a preview, not the fast viewer</b></summary>

**Interactive View** in *Visualization* is meant to inspect the cube **before** exporting it. The scenes therefore load slowly, and the speed depends on your local machine.

For fast viewing and animations, export the cube first and open it in the **Data Cube Editor**.

</details>

<details>
<summary><b>Required bands for super-resolution in <i>3. Analysis Ready Data Cube Tools</i></b></summary>

Each super-resolution model needs a fixed set of bands, so pick them here while building. The order you tick them in does not matter, and the cube must be built at **10 m** for all three models.

Spectral indices are **recalculated** from the super-resolved bands, not resampled. So an index can only be kept if the bands it needs are part of the model's output.

**10-m RGBN to 2.5-m**
- must: `blue`, `green`, `red`, `nir`
- optional: 10-m indices - `ndvi`, `ndwi`, `savi`, `evi`

**10-m and 20-m Full Spectral to 2.5-m**
- must: `blue`, `green`, `red`, `nir`, `rededge1`, `rededge2`, `rededge3`, `nir08`, `swir16`, `swir22`
- optional: 10-m and 20-m indices - any of `ndvi`, `ndwi`, `savi`, `evi`, `ndmi`, `nbr`, `mndwi`, `ndbi`, `ndre1`, `ndsi`

**20-m Bands to 10-m**
- must: the same ten bands as above (the four 10-m bands guide the six 20-m ones)
- optional: 10-m and 20-m indices - the same list

Missing bands are reported when the tool starts. You can add them afterwards with **Data Cube Editor -> Update Data Cube (date and/or band)** instead of rebuilding.

</details>

In [ ]:
from stac2cube import datacube_builder

In [ ]:
builder = datacube_builder()

# 2. Data Cube Editor

Open an existing cube and work on it. Tools are **chained**: tick as many as you like, then click **Edit data cube** once, they run in the order shown in the interface. **Reset to loaded cube** undoes everything.

<details>
<summary><b>What is in this interface?</b></summary>

Expand a tool to see what it does. The interface is laid out in the same order.

---

**Source**

<details>
<summary><b>Open a cube</b> - load a NetCDF or Zarr cube</summary>

Loads the file and lets you choose which layer to work on (the time series, or a temporal composite saved with it).

</details>

<details>
<summary><b>Mosaic several cubes</b> - join cubes side by side into one</summary>

Puts cubes covering neighbouring areas onto one grid, for an area that was built in parts. Nothing is downloaded and nothing is recalculated. You choose what happens where cubes overlap, the output projection and pixel size, and which dates, bands and layers to keep.

</details>

---

**Extend**

<details>
<summary><b>Update Data Cube (date and/or band)</b> - fetch the dates and bands the cube is missing</summary>

Queries the catalog for scenes and bands the loaded cube does not have yet. The cube's cloud/shadow masking strategy is restored from its attributes, so new data is treated exactly like the stored data.

It replaces the current working result, so run it first, on its own, then continue editing.

</details>

---

**Edit**

*1 - Select a subset*

<details>
<summary><b>Slice Data Cube</b> - keep only the dates and/or bands you need</summary>

Filter by time, by band, or both.

</details>

<details>
<summary><b>Filter by Cloud Coverage</b> - drop time steps above a cloud percentage</summary>

Uses the `cloud_percentage` coordinate already stored in the cube. This is **not** a new cloud detection and **not** the STAC `max_cc` filter.

Best used before clipping and before composites: cloud percentages are not recalculated in the editor after clipping.

</details>

<details>
<summary><b>Filter by Scene Coverage</b> - drop scenes that image only part of your area</summary>

Uses the `scene_coverage` coordinate (the share of the area each scene actually images). Removes across-track / swath-edge scenes and faulty or partially missing acquisitions.

</details>

<details>
<summary><b>Mask Clouds with Binary Masking File</b> - apply a binary cloud mask</summary>

Masks the loaded cube with a binary mask file (1 = cloud, 0 = clear) that comes from the same cube, e.g. the one written by **Build Cloud Mask Cube**. It is matched to the cube's current dates, so it still works after slicing.

</details>

*2 - Change the geometry*

<details>
<summary><b>Clip Raster</b> - cut to a polygon or a bbox</summary>

Polygon formats: `gpkg`, `geojson`, `kml`, `kmz`, `shp` - geographic (WGS84) or projected. Alternatively a WGS84 bbox list `[xmin, ymin, xmax, ymax]`.

</details>

<details>
<summary><b>Reproject Data Cube</b> - warp the cube into another projection</summary>

The target CRS must be projected and metre-based. Class layers (`scl`, QA, `cloud_mask_*`) always use nearest resampling, whatever you select.

Reprojection resamples: pixel values and the pixel grid both change and the step cannot be undone exactly. Reproject once, and expect empty corners in the result.

</details>

*3 - Add new bands*

<details>
<summary><b>Calculate Spectral Indices</b> - append indices as new bands</summary>

Computed from the bands already in the cube. Which indices are available depends on the mission, and each one needs specific bands (`ndvi` needs `red` and `nir`). Missing bands are reported in the Status box, and indices already present are skipped.

</details>

*4 - Collapse the time axis*

<details>
<summary><b>Temporal Composites</b> - mean, median, min, max or std over time</summary>

Over the whole series, per month, per year, or over a custom period you define yourself (e.g. a spring window, repeated in every year the cube covers). Each composite is added as its own layer next to the time series.

Untick **Keep the full time series** to keep only the composites.

</details>

---

**Result, view, export**

<details>
<summary><b>Visualization</b> - interactive viewer and time-series animation</summary>

Explore the cube scene by scene (RGB and false-colour presets, single bands in grey levels, or your own R/G/B combination), and render the whole time series to an animated GIF.

</details>

<details>
<summary><b>Export Options</b> - NetCDF, Zarr or Cloud Optimized Geotiffs</summary>

- **NetCDF (`.nc`)** - one single file, works with every ARD cube tool, easy to share. A solid default.
- **Zarr (`.zarr`)** - a chunked folder, also works with every ARD cube tool, quickest for very large cubes. Zip it to share it.
- **Cloud Optimized Geotiffs** - a folder with one Geotiff per date, ready to drag into QGIS. **Not** accepted by the ARD cube tools.

</details>

---

**Side outputs**

<details>
<summary><b>Build Cloud Mask Cube</b> - write the cube's binary cloud mask to a separate file</summary>

Builds the SCL-based binary cloud mask of the loaded cube (1 = cloud, 0 = clear) as its own file. Use it to co-register or mask a cube that is not masked yet. It re-queries the catalog, so the cube's original file is needed, and it does not change the working result.

</details>

</details>

In [ ]:
from stac2cube import datacube_editor

In [ ]:
editor = datacube_editor()

# 3. Analysis Ready Data Cube Tools

Load a NetCDF or Zarr cube, then run one or more of the three tools below.

<details>
<summary><b>What is in this interface?</b></summary>

Load a NetCDF or Zarr data cube, then run one or more of the three tools. Expand a tool to see what it does.

---

<details>
<summary><b>1) Cloud and Shadow Masking Data Cube</b> - probability-based masking (s2cloudless)</summary>

Builds cloud masks with your own thresholds and masks the Sentinel-2 cube with them.

- **a) Fully automated workflow** - the whole chain in one run.
- **b) Manual** - step by step: i) build the cloud masking data cube, ii) optionally generate masks at other thresholds from the probability map, iii) optionally build a cloud shadow mask, iv) mask out the data cube.

</details>

<details>
<summary><b>2) Co-register Data Cube</b> - reduce scene-to-scene misalignment</summary>

Aligns the time series within the cube (not globally). You choose the reference scene, optional filters and advanced settings; the Spectral Profiler helps to judge the result.

</details>

<details>
<summary><b>3) Super-resolve Data Cube</b> - 10-m and 20-m bands to 2.5 m, or 20-m bands to 10 m</summary>

Sharpens the time series to a finer pixel size. The result is no longer 10 m, so it cannot be co-registered afterwards.

</details>

</details>

<details>
<summary><b>💡 GOOD TO KNOW</b></summary>

- For the best outcome, follow the order **Cloud Masking -> Co-registration -> Super-resolution**, but it is not mandatory, depending on the use case.

- A super-resolved data cube cannot be co-registered (it has to be 10 metres).

- Co-registration needs one of the following, otherwise cloud pixels will significantly affect the procedure:

  a) cloud masking, either during the initial data cube generation or with the cloud masking tool above, **or**

  b) a binary cloud mask file, which can be generated in **Data Cube Editor -> Build Cloud Mask Cube**.

</details>

In [ ]:
from stac2cube import ard_cube_tools

In [ ]:
ard_tools = ard_cube_tools()

# Troubleshooting

<details>
<summary><b>1) Output shows <code>Could not render content for 'application/vnd.jupyter.widget-view+json'</code></b></summary>

```
Could not render content for 'application/vnd.jupyter.widget-view+json'
{"model_id":"...","version_major":2,"version_minor":0}
```

This is normal after reopening a session. The interface (ipywidget) lives in the kernel, but once the kernel is gone its visual output cannot be rebuilt from the saved file, so only this leftover placeholder from the previous session is shown.

**Fix:** re-run the cell that creates the tool (e.g. `builder = datacube_builder()`) and the interface will show up again.

</details>

<details>
<summary><b>2) The interface does not show up, only "..." (three dots) appear</b></summary>

This happens sometimes. Just restart the software (restart the kernel / VS Code / Jupyter) and run the cell again.

</details>

<details>
<summary><b>3) Output shows <code>ERROR 1: PROJ: proj_create_from_database</code></b></summary>

```
ERROR 1: PROJ: proj_create_from_database: Open of /path/share/proj failed
```

This is a known message and is harmless. It does not change anything in the pipeline, you can safely ignore it.

</details>

<details>
<summary><b>4) Duplicated output (the same interface or output appears more than once)</b></summary>

This is something happening in VS Code. Restarting the software or the device usually solves it.

</details>